# Zedboard Level-Safe Interrupt Service with Heartbeat LED
This notebook initializes dual-register AXI GPIO interrupts for switches and buttons on the Zedboard. It contains a 3-cycle consecutive-sampling software debounce mechanism and a background heartbeat task to flash an status indicator LED.

In [1]:
#!/usr/bin/env python3
import sys
import time
import asyncio
import logging
from pynq import Overlay
from pynq import Interrupt

# Configure logging to output directly into the notebook cell
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger("hardware_daemon")
                           
GPIO0_BASE = 0x41200000  # Switches (CH2 in) -> LEDs (CH1 out)
GPIO1_BASE = 0x41210000  # Buttons  (CH1 in)
    
# AXI GPIO Register Offsets
GPIO_DATA = 0x0
GPIO_TRI = 0x4
GPIO2_DATA = 0x8
GPIO2_TRI = 0xC
GIER = 0x11C
IP_IER = 0x128
IP_ISR = 0x120

# Debounce Constants & Hardware Masks
DEBOUNCE_MS = 20
LED_SWITCH_MASK = 0x7F     # Masking bits 0-6 for switches
LED_HEARTBEAT_MASK = 0x80  # Top LED (Bit 7) used for heartbeat
BTN_MASK = 0x1F

# Global State Shadows
led_state = 0x00
current_switches = 0
current_buttons = 0
prev_btn_state = 0


## 1. Bitstream Initialization & Core Registrations

In [2]:
print("[BOOT.PY] Initializing dual-register interrupt service...")

try:
    ol = Overlay("/home/xilinx/ycorrales/zedboard.bit")
except Exception as e:
    print(f"[BOOT.PY] CRITICAL: Failed parsing bitstream layout: {e}")
    sys.exit(1)
    
gpio0 = ol.axi_gpio_0
gpio1 = ol.axi_gpio_1
intc = ol.axi_intc_0

# Core 0 Layout Configuration (Switches & LEDs)
gpio0.write(GPIO_TRI, 0x00)   # CH1 = output (LEDs)
gpio0.write(GPIO2_TRI, 0xFF)  # CH2 = input  (switches)
gpio0.write(GIER, 0x80000000) # Enable Global Interrupts
gpio0.write(IP_IER, 0x2)      # Enable Channel 2 (Switches)

# Core 1 Layout Configuration (Buttons)\n",
gpio1.write(GPIO_TRI, 0xFF)   # CH1 = input (buttons)
gpio1.write(GIER, 0x80000000) # Enable Global Interrupts
gpio1.write(IP_IER, 0x01)     # Enable Channel 1 (Buttons)
    
# AXI Interrupt Controller (INTC) Setup
intc.write(0x10, 0x3)        # Enable interrupt lines
intc.write(0x1C, 0x3)        # Global interrupt enable

# Bind PYNQ UIO file descriptors to target lines
switch_interrupt = Interrupt("xlconcat_0/In1")
button_interrupt = Interrupt("xlconcat_0/In0")


[BOOT.PY] Initializing dual-register interrupt service...


## 2. Hardware Helper & Application Path Modules

In [5]:
def led_apply(clear_mask: int, set_bits: int):
    """
    Merge bits into LED shadow register and push to hardware.
    """
        
    global led_state
    clear_mask &= 0xFF
    set_bits &= 0xFF
    
    inverted_mask = (~clear_mask) & 0xFF
    cleared_state = led_state & inverted_mask
    applied_bits = set_bits & clear_mask

    led_state = (cleared_state | applied_bits) & 0xFF
    gpio0.write(GPIO_DATA, led_state)

async def process_switch_path():
    global current_switches
    sample = 0
    prev = 0xFFFFFFFF
    stable = 0
    
    while stable < 3:
        await asyncio.sleep(DEBOUNCE_MS / 1000.0)
        sample = gpio0.read(GPIO2_DATA)
        if sample == prev:
            stable += 1
        else:
            stable = 0
        prev = sample

    gpio0.write(IP_ISR, 0x3)
    current_switches = sample
    led_apply(LED_SWITCH_MASK, sample & LED_SWITCH_MASK)
    intc.write(0x0C, 0x2)
    logger.info(f"SW IRQ: switches=0x{sample & 0xFF:02X}")

async def process_button_path():
    global current_buttons, prev_btn_state
    sample = 0
    prev = 0xFFFFFFFF
    stable = 0

    while stable < 3:
        await asyncio.sleep(DEBOUNCE_MS / 1000.0)
        sample = gpio1.read(GPIO_DATA) & BTN_MASK
        if sample == prev:
            stable += 1
        else:
            stable = 0
        prev = sample

    gpio1.write(IP_ISR, 0x3)
    current_buttons = sample
    pressed = sample & ~prev_btn_state
    prev_btn_state = sample
    intc.write(0x0C, 0x1)
 
    if pressed > 0:
        logger.info(f"BTN IRQ: pressed edge detected=0x{pressed:02X}")


## 3. Background LED Heartbeat Task

In [6]:
async def heartbeat_task():
    """
    Toggles the configured heartbeat bit mask dynamically at a 500ms cycle.
    """

    print(">>> Heartbeat Monitor Active...")
    is_on = False
    while True:
        is_on = not is_on
        set_bits = LED_HEARTBEAT_MASK if is_on else 0x00
        led_apply(LED_HEARTBEAT_MASK, set_bits)
        await asyncio.sleep(0.5)


## 4. Concurrent Event Loop Execution Engine

In [9]:
async def watch_switches():
    print(">>> Level-Safe Switch Listener Active...")
    while True:
        await switch_interrupt.wait()
        await process_switch_path()

async def watch_buttons():
    print(">>> Level-Safe Button Listener Active...")
    while True:
        await button_interrupt.wait()
        await process_button_path()
    
async def main():
    print("[BOOT.PY] Scheduling asynchronous tasks and loops concurrently...")
    await asyncio.gather(
        watch_switches(),
        watch_buttons(),
        heartbeat_task()
    )
    
try:
    await main()
except KeyboardInterrupt:
    print("\\n[BOOT.PY] Gracefully shutting down listeners...")


[BOOT.PY] Scheduling asynchronous tasks and loops concurrently...
>>> Level-Safe Switch Listener Active...
>>> Level-Safe Button Listener Active...
>>> Heartbeat Monitor Active...


2025-05-03 23:21:20,586 - INFO - SW IRQ: switches=0x08
2025-05-03 23:21:21,956 - INFO - SW IRQ: switches=0x00


NameError: name 'intc0' is not defined

2025-05-03 23:21:36,114 - INFO - SW IRQ: switches=0x00
2025-05-03 23:21:37,674 - INFO - SW IRQ: switches=0x04
2025-05-03 23:21:38,959 - INFO - SW IRQ: switches=0x0C
2025-05-03 23:21:40,108 - INFO - SW IRQ: switches=0x1C
